# Model 1 feature engineering (JDSearch)

Builds one row per (slate, candidate) with the Model 1 features and saves them to `features/`.

**Inputs** (in `data/`, not committed to git): `train_sampled.parquet`, `val.parquet`, `test.parquet`, `product_metadata_clean.parquet` from the preprocessing notebook.

**Outputs:** `features/features_train.parquet`, `features_val.parquet`, `features_test.parquet`.

**Run order:** Restart kernel, then run all cells top to bottom.

**Design notes**
- Train uses the negative-sampled slates; val and test keep full slates.
- Popularity is built from **train history only**, so val/test information never leaks into features.
- History events within `MIN_AGE_DAYS` of the query are ignored: they include the slate's own click/cart/order events (checked in the diagnostics notebook).
- Each action type is only used inside its observation window (CLICK 7 days, CART 90 days, ORD 365 days).

## 0. Imports and config

In [1]:
import os, math
from collections import Counter, defaultdict
import numpy as np, pandas as pd, pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

SEP = "\x18"
WEIGHT = {"CLICK": 1, "FLW": 2, "CART": 5, "ORD": 10}
W_LIST = [1, 3, 7]               # recent weighted-affinity windows in days (14 dropped: clicks only exist for ~7 days)
TYPE_WIN = {"CLICK": 7, "CART": 90, "ORD": 365}   # each action type only within its valid observation window
META_COLS = ["wid", "name", "brand_id", "cate_id_3", "cate_id_4"]
MIN_AGE_DAYS = 0.01             # ignore history events within ~15 min of the query (they include the slate's own events => label leak)
PROFILE_PER_BATCH = 3            # products sampled per category per batch
PROFILE_CAP = 200                # max products per category in the profile

DATA = "data"       # folder with the cleaned parquet files (git-ignored)
OUT = "features"    # output folder for the feature tables (git-ignored)
os.makedirs(OUT, exist_ok=True)

## 1. Loader

In [2]:
def load(path):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path).to_pandas(ignore_metadata=True)

## 2. Explode list columns into one row per candidate / history action

In [3]:
def explode_cand(df):
    lens = df["candidate_wid_list"].map(len).to_numpy()
    out = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "wid": np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64),
        "label": np.concatenate(df["candidate_label_list"].to_numpy()).astype(np.int8),
        "cand_has_meta": np.concatenate(df["candidate_has_metadata"].to_numpy()).astype(np.int8),
    })
    assert len(out) == lens.sum()
    return out


def explode_hist(df):
    lens = df["history_wid_list"].map(len).to_numpy()
    typ = np.concatenate(df["history_type_list"].to_numpy()).astype(str)
    days = np.concatenate(df["history_days_before"].to_numpy()).astype(np.float32)
    assert len(typ) == len(days) == lens.sum(), "history list lengths disagree"
    out = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "wid": np.concatenate(df["history_wid_list"].to_numpy()).astype(np.int64),
        "typ": typ, "days": days})
    out["w"] = out["typ"].map(WEIGHT).fillna(1).astype(np.int16)
    return out

## 3. Metadata scan and category profile
One streaming pass over the catalogue, plus the text statistics used by the cosine similarities.

In [4]:
def scan_metadata(path, cand_wids, hist_wids):
    """One streaming pass over the catalogue.
    Returns (meta, profile_rows):
      meta: wid -> brand_id, cate_id_4, (name only for candidate wids)
      profile_rows: sample of (cate_id_4, name) used for the category profile."""
    cand_wids = np.unique(cand_wids)
    need = np.unique(np.concatenate([cand_wids, hist_wids]))
    keep, prof = [], []
    pf = pq.ParquetFile(path)
    for b in pf.iter_batches(batch_size=1_000_000, columns=META_COLS):
        d = b.to_pandas(ignore_metadata=True)
        # category-profile sample (before filtering): a few products per cate_id_4
        ok = d[d["cate_id_4"] >= 0]
        prof.append(ok.groupby("cate_id_4", sort=False).head(PROFILE_PER_BATCH)[["cate_id_4", "name"]])
        m = np.isin(d["wid"].to_numpy(), need)
        d = d[m]
        is_cand = np.isin(d["wid"].to_numpy(), cand_wids)
        d.loc[~is_cand, "name"] = None
        keep.append(d[["wid", "name", "brand_id", "cate_id_4"]])
    meta = pd.concat(keep).drop_duplicates("wid").set_index("wid")
    for c in ["brand_id", "cate_id_4"]:
        meta[c] = meta[c].where(meta[c] >= 0)          # -1 -> missing
    prof = pd.concat(prof)
    prof = prof.groupby("cate_id_4", sort=False).head(PROFILE_CAP)
    return meta, prof


def build_cate_profile(prof):
    """Text statistics for the cosine similarities, from the sampled catalogue names.
    returns dict:
      cate: cate_id_4 -> {term: tf*idf weight} (top 300 terms)      -> category vector
      norm: cate_id_4 -> L2 norm of that vector
      idf : term -> product-level idf (for the name cosine); idf_default for unseen terms"""
    tf = defaultdict(Counter); n = Counter(); df_prod = Counter(); n_prod = 0
    for cate, name in zip(prof["cate_id_4"].to_numpy(), prof["name"]):
        if name is None or len(name) == 0:
            continue
        terms = set(map(str, name))
        n[cate] += 1; n_prod += 1; df_prod.update(terms)
        for t in terms:
            tf[cate][t] += 1
    df_cate = Counter()
    for cate in tf:
        df_cate.update(tf[cate].keys())
    N = max(len(tf), 1)
    cate_w, norm = {}, {}
    for cate, cnt in tf.items():
        w = {t: (c / n[cate]) * math.log((1 + N) / (1 + df_cate[t])) for t, c in cnt.items()}
        top = dict(sorted(w.items(), key=lambda kv: -kv[1])[:300])
        nrm = math.sqrt(sum(v * v for v in top.values()))
        if nrm > 0:
            cate_w[cate], norm[cate] = top, nrm
    idf = {t: math.log((1 + n_prod) / (1 + d)) + 1 for t, d in df_prod.items()}
    return {"cate": cate_w, "norm": norm, "idf": idf, "idf_default": math.log(1 + n_prod) + 1}

## 4. Feature functions

In [5]:
def query_terms(q):
    """Cleaned data stores the query as an array of term-id strings; raw data as one \x18-joined string."""
    if isinstance(q, str):
        return [t for t in q.split(SEP) if t != ""]
    return [str(t) for t in q if str(t) != ""]


def add_base(c, df):
    qterms = df["query"].map(query_terms)
    q = pd.DataFrame({"slate_id": df["slate_id"].to_numpy(),
                      "current_query_length": qterms.map(len).to_numpy(), "qterms": qterms.to_numpy()})
    return c.merge(q, on="slate_id", how="left")


def build_popularity(tr):
    """TRAIN-history-only popularity: all actions, ORD only (full year), CART only (90 days)."""
    wid = np.concatenate(tr["history_wid_list"].to_numpy())
    typ = np.concatenate(tr["history_type_list"].to_numpy()).astype(str)
    days = np.concatenate(tr["history_days_before"].to_numpy()).astype(np.float32)
    ok = days >= MIN_AGE_DAYS                     # drop events at the query moment (label events)
    wid, typ = wid[ok], typ[ok]
    return {"all": pd.Series(wid).value_counts(),
            "ord": pd.Series(wid[typ == "ORD"]).value_counts(),
            "cart": pd.Series(wid[typ == "CART"]).value_counts()}


def add_popularity(c, pop):
    c["candidate_popularity_count"] = c["wid"].map(pop["all"]).fillna(0).astype(np.float32)
    c["candidate_log_popularity"] = np.log1p(c["candidate_popularity_count"])
    c["candidate_log_pop_ord"] = np.log1p(c["wid"].map(pop["ord"]).fillna(0)).astype(np.float32)
    c["candidate_log_pop_cart"] = np.log1p(c["wid"].map(pop["cart"]).fillna(0)).astype(np.float32)
    return c


def add_type_features(c, h):
    """Per-action-type counts inside each type's own window, for the same item / brand / cate_4,
    plus days since the last CART / ORD of the same item / brand (NaN = never)."""
    for key, short in [("wid", "item"), ("brand_id", "brand"), ("cate_id_4", "cate")]:
        hk = h[h[key].notna()]
        missing = c[key].isna().to_numpy()
        base = pd.DataFrame({"slate_id": c["slate_id"].to_numpy(),
                             key: c[key].fillna(-1).astype(np.int64).to_numpy()})
        for typ, win in TYPE_WIN.items():
            s = hk[(hk["typ"] == typ) & (hk["days"] <= win)]
            g = s.groupby(["slate_id", key])["days"].agg(["size", "min"]).reset_index()
            g[key] = g[key].astype(np.int64)
            m = base.merge(g, on=["slate_id", key], how="left")
            cnt = np.nan_to_num(m["size"].to_numpy(dtype=float), nan=0.0)
            cnt[missing] = np.nan
            c[f"{short}_{typ.lower()}_cnt_{win}d"] = cnt.astype(np.float32)
            if typ in ("CART", "ORD") and short in ("item", "brand"):
                d = m["min"].to_numpy(dtype=float); d[missing] = np.nan
                c[f"{short}_days_since_{typ.lower()}"] = d.astype(np.float32)
    return c


def add_recent_affinity(c, h):
    for key, short in [("brand_id", "brand"), ("cate_id_4", "cate")]:
        hk = h[h[key].notna()]
        ck = c[key].fillna(-1).astype(np.int64)
        for W in W_LIST:
            s = hk[hk["days"] <= W].groupby(["slate_id", key])["w"].sum().rename("v").reset_index()
            s[key] = s[key].astype(np.int64)
            tmp = pd.DataFrame({"slate_id": c["slate_id"].to_numpy(), key: ck.to_numpy()})
            v = tmp.merge(s, on=["slate_id", key], how="left")["v"].to_numpy()
            v = np.where(np.isnan(v), 0.0, v)
            v = np.where(c[key].isna().to_numpy(), np.nan, v)      # NaN only if candidate lacks metadata
            c[f"recent_{short}_affinity_{W}d"] = v.astype(np.float32)
    return c


def add_similarity(c, prof_w):
    """Cosine similarities between the query and (a) the candidate's name, (b) its cate_id_4 profile.
    Query is a binary term vector; name uses idf-weighted terms; category uses the tf*idf profile."""
    idf, d0 = prof_w["idf"], prof_w["idf_default"]
    wt2 = lambda t: idf.get(t, d0) ** 2
    qsets = [frozenset(map(str, q)) for q in c["qterms"].to_numpy()]
    nsets = [frozenset(map(str, n)) if n is not None and len(n) else frozenset() for n in c["name"]]
    qnorm_cache = {}
    name_sim = np.full(len(c), np.nan, dtype=np.float32)
    for i, (qs, ns) in enumerate(zip(qsets, nsets)):
        if not qs or not ns:
            continue
        if qs not in qnorm_cache:
            qnorm_cache[qs] = math.sqrt(sum(wt2(t) for t in qs))
        nn = math.sqrt(sum(wt2(t) for t in ns))
        if qnorm_cache[qs] > 0 and nn > 0:
            name_sim[i] = sum(wt2(t) for t in (qs & ns)) / (qnorm_cache[qs] * nn)
    c["query_candidate_name_sim"] = name_sim
    cache, out = {}, np.full(len(c), np.nan, dtype=np.float32)
    cw, cn = prof_w["cate"], prof_w["norm"]
    for i, (qs, cate) in enumerate(zip(qsets, c["cate_id_4"].to_numpy())):
        if not qs or np.isnan(cate) or cate not in cw:
            continue
        k = (qs, cate)
        if k not in cache:
            p = cw[cate]
            cache[k] = sum(p.get(t, 0.0) for t in qs) / (math.sqrt(len(qs)) * cn[cate])
        out[i] = cache[k]
    c["query_candidate_cate4_sim"] = out
    return c


def build_features(df, meta, pop, prof_w):
    c = explode_cand(df)
    c = c.join(meta[["name", "brand_id", "cate_id_4"]], on="wid")
    # trust the teammate's flag as the definition of 'has metadata'
    for col in ["name", "brand_id", "cate_id_4"]:
        if col == "name":
            c.loc[c["cand_has_meta"] == 0, col] = None
        else:
            c.loc[c["cand_has_meta"] == 0, col] = np.nan
    h = explode_hist(df)
    h = h[h["days"] >= MIN_AGE_DAYS]
    h = h.join(meta[["brand_id", "cate_id_4"]], on="wid")
    c = add_base(c, df)
    c = add_popularity(c, pop)
    c = add_recent_affinity(c, h)
    c = add_type_features(c, h)
    c = add_similarity(c, prof_w)
    feats = (["current_query_length", "cand_has_meta", "candidate_popularity_count", "candidate_log_popularity"]
             + [f"recent_{s}_affinity_{W}d" for s in ("brand", "cate") for W in W_LIST]
             + ["candidate_log_pop_ord", "candidate_log_pop_cart"]
             + [f"{k}_{t.lower()}_cnt_{w}d" for k in ("item", "brand", "cate") for t, w in TYPE_WIN.items()]
             + [f"{k}_days_since_{t}" for k in ("item", "brand") for t in ("cart", "ord")]
             + ["query_candidate_name_sim", "query_candidate_cate4_sim"])
    out = c[["slate_id", "wid", "label"] + feats].rename(columns={"cand_has_meta": "candidate_has_metadata"})
    return out

## 5. Load the splits and check alignment

In [6]:
splits = {"train": load(f"{DATA}/train_sampled.parquet"),
          "val": load(f"{DATA}/val.parquet"),
          "test": load(f"{DATA}/test.parquet")}
for k, d in splits.items():
    print(k, len(d), "slates")
print("total slates:", sum(len(d) for d in splits.values()))
tr = splits["train"]

# every list column must line up within a row
for k, d in splits.items():
    assert (d["candidate_wid_list"].map(len) == d["candidate_label_list"].map(len)).all()
    assert (d["candidate_wid_list"].map(len) == d["candidate_has_metadata"].map(len)).all()
    assert (d["history_wid_list"].map(len) == d["history_type_list"].map(len)).all()
    assert (d["history_wid_list"].map(len) == d["history_days_before"].map(len)).all()
print("list lengths aligned")

train 139882 slates
val 16924 slates
test 17025 slates
total slates: 173831
list lengths aligned


## 6. Train-only popularity

In [7]:
pop = build_popularity(tr)
print({k: len(v) for k, v in pop.items()}, "distinct products by action type (train history)")

{'all': 4718341, 'ord': 1731016, 'cart': 3141443} distinct products by action type (train history)


## 7. Scan the catalogue and build the category profile
Slow (a few minutes) and memory-heavy. Only needs to run once per kernel session.

In [8]:
cand_w = np.concatenate([np.concatenate(d["candidate_wid_list"].to_numpy()) for d in splits.values()])
hist_w = np.unique(np.concatenate([np.concatenate(d["history_wid_list"].to_numpy()) for d in splits.values()]))
meta, prof = scan_metadata(f"{DATA}/product_metadata_clean.parquet", cand_w, hist_w)
prof_w = build_cate_profile(prof)
print(len(meta), "products with metadata matched;", len(prof_w["cate"]), "categories profiled;",
      len(prof_w["idf"]), "terms with idf")

7784366 products with metadata matched; 7137 categories profiled; 186207 terms with idf


## 8. Build and save features

In [9]:
feat = {}
for name, df in splits.items():
    f = build_features(df, meta, pop, prof_w)
    f.to_parquet(f"{OUT}/features_{name}.parquet", index=False)
    feat[name] = f
    print(name, f.shape, f"purchase rate {f['label'].eq(3).mean():.4%}")

train (2988641, 30) purchase rate 0.5618%
val (1511179, 30) purchase rate 0.1311%
test (1500706, 30) purchase rate 0.1376%


## 9. Sanity checks

### 9.1 Missing rates
The affinity, brand/category and similarity columns should be missing only for candidates without metadata (about 0.45%). The `days_since_*` columns are mostly empty by design (NaN = never).

In [10]:
for s, f in feat.items():
    print("=====", s, f.shape, "slates:", f.slate_id.nunique())
    print(f.isna().mean().round(4).to_string())

===== train (2988641, 30) slates: 139882
slate_id                      0.0000
wid                           0.0000
label                         0.0000
current_query_length          0.0000
candidate_has_metadata        0.0000
candidate_popularity_count    0.0000
candidate_log_popularity      0.0000
recent_brand_affinity_1d      0.0045
recent_brand_affinity_3d      0.0045
recent_brand_affinity_7d      0.0045
recent_cate_affinity_1d       0.0045
recent_cate_affinity_3d       0.0045
recent_cate_affinity_7d       0.0045
candidate_log_pop_ord         0.0000
candidate_log_pop_cart        0.0000
item_click_cnt_7d             0.0000
item_cart_cnt_90d             0.0000
item_ord_cnt_365d             0.0000
brand_click_cnt_7d            0.0045
brand_cart_cnt_90d            0.0045
brand_ord_cnt_365d            0.0045
cate_click_cnt_7d             0.0045
cate_cart_cnt_90d             0.0045
cate_ord_cnt_365d             0.0045
item_days_since_cart          0.9952
item_days_since_ord           0.99

### 9.2 Consistency checks

In [11]:
# nested windows must be monotone (1d <= 3d <= 7d)
for s, f in feat.items():
    for short in ("brand", "cate"):
        cols = [f"recent_{short}_affinity_{W}d" for W in W_LIST]
        v = f[cols].fillna(0).to_numpy()
        assert (np.diff(v, axis=1) >= 0).all(), (s, short)
print("nested windows OK")

# missing metadata must give missing similarity / affinity
for s, f in feat.items():
    nm = f["candidate_has_metadata"].eq(0)
    assert f.loc[nm, "query_candidate_name_sim"].isna().all()
    assert f.loc[nm, "recent_brand_affinity_3d"].isna().all()
    print(s, "no-metadata share:", round(nm.mean() * 100, 3), "%")

nested windows OK
train no-metadata share: 0.454 %
val no-metadata share: 0.449 %
test no-metadata share: 0.459 %


### 9.3 Similarity features: distribution and purchase gap

In [12]:
f = feat["val"]
for c in ["query_candidate_name_sim", "query_candidate_cate4_sim"]:
    print(c, f[c].describe().round(4).to_dict())
    print("  share exactly 0:", (f[c] == 0).mean().round(3))
print(f.groupby(f.label.eq(3))[["query_candidate_name_sim", "query_candidate_cate4_sim"]].mean())

query_candidate_name_sim {'count': 1504397.0, 'mean': 0.2427, 'std': 0.1386, 'min': 0.0, '25%': 0.1555, '50%': 0.2506, '75%': 0.3319, 'max': 1.0}
  share exactly 0: 0.105
query_candidate_cate4_sim {'count': 1504397.0, 'mean': 0.2221, 'std': 0.1806, 'min': 0.0, '25%': 0.0493, '50%': 0.2006, '75%': 0.3682, 'max': 0.83}
  share exactly 0: 0.113
       query_candidate_name_sim  query_candidate_cate4_sim
label                                                     
False                  0.242699                   0.222056
True                   0.267706                   0.251694


### 9.4 Single-feature AUC on val
Pooled over all slates, so ranking features can look weaker than they are. Popularity should be the strongest; the similarity and brand/category order-history features should beat 0.5.

In [13]:
f = feat["val"]; y = f["label"].eq(3).astype(int)
for c in [c for c in f.columns if c not in ("slate_id", "wid", "label")]:
    x = f[c].fillna(f[c].median())
    print(f"{c:35s} AUC = {roc_auc_score(y, x):.3f}")

current_query_length                AUC = 0.468
candidate_has_metadata              AUC = 0.501
candidate_popularity_count          AUC = 0.810
candidate_log_popularity            AUC = 0.810
recent_brand_affinity_1d            AUC = 0.505
recent_brand_affinity_3d            AUC = 0.503
recent_brand_affinity_7d            AUC = 0.503
recent_cate_affinity_1d             AUC = 0.493
recent_cate_affinity_3d             AUC = 0.486
recent_cate_affinity_7d             AUC = 0.481
candidate_log_pop_ord               AUC = 0.802
candidate_log_pop_cart              AUC = 0.797
item_click_cnt_7d                   AUC = 0.515
item_cart_cnt_90d                   AUC = 0.517
item_ord_cnt_365d                   AUC = 0.525
brand_click_cnt_7d                  AUC = 0.502
brand_cart_cnt_90d                  AUC = 0.519
brand_ord_cnt_365d                  AUC = 0.558
cate_click_cnt_7d                   AUC = 0.482
cate_cart_cnt_90d                   AUC = 0.504
cate_ord_cnt_365d                   AUC 

### 9.5 No position leak
Candidates are shuffled inside each slate, so the first position should not be special.

In [14]:
first = feat["val"].groupby("slate_id").head(1)
print("purchase rate overall:", feat["val"]["label"].eq(3).mean().round(5),
      "| first position:", first["label"].eq(3).mean().round(5))

purchase rate overall: 0.00131 | first position: 0.00183
